# 12 · Circularity & temporal-leakage reference

> **What this notebook is for.** It is the **reference you consult before trusting any
> predictor's verdict on a CFTR variant** — including the future `predict/` pipeline.
> It answers two questions: *(1) did the tool learn from the answer key you are testing
> it against (circularity)?* and *(2) could the tool have already seen this exact
> variant's clinical label because it was reported before the tool was trained
> (temporal leakage)?*

Nothing here is a score you quote — it is the set of **caveats** you attach to scores
from notebooks 02–11.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1. Two kinds of predictor — and why it decides everything

A variant-effect predictor is only independent evidence *if it did not learn from
the answer key you are testing it against.*

- **Unsupervised** (AlphaMissense, EVE, ESM1b): trained on **protein sequences /
  evolutionary alignments only** — never on clinical pathogenic/benign labels.
  Comparing them to ClinVar is (largely) fair.
- **Semi-supervised** (PrimateAI, CADD): trained on a *proxy* (common/primate
  variants = "probably benign", simulated variants = "probably not observed).
  Some leakage risk.
- **Supervised** (REVEL): trained **directly on curated pathogenic/benign
  variants** whose lineage overlaps ClinVar/HGMD. Comparing REVEL to ClinVar is
  **partly circular** — high agreement is partly memorisation, and a
  "disagreement" may just be label noise, not new evidence.

In [2]:
reg = (pd.DataFrame(tk.TOOL_REGISTRY).T
         .reset_index().rename(columns={"index":"tool"})
         [["tool","kind","learning","circularity","signal"]])
# order from safest to most circular for ClinVar benchmarking
order = {"low":0,"low-for-clinvar":0,"medium":1,"HIGH (label lineage overlaps ClinVar/HGMD)":2}
reg["rank"] = reg["circularity"].map(lambda c: order.get(c, 1))
reg.sort_values("rank")[["tool","kind","learning","circularity"]].reset_index(drop=True)

,tool,kind,learning,circularity
0,AlphaMissense,missense,unsupervised,low
1,EVE,missense,unsupervised,low
2,ESM1b,missense,unsupervised,low
3,SpliceAI,splice,"supervised (on GTEx splice junctions, not clin...",low-for-clinvar
4,Pangolin,splice,supervised (on splice usage across species/tis...,low-for-clinvar
5,PrimateAI,missense,semi-supervised,medium
6,CADD,general,semi-supervised,medium
7,REVEL,missense,supervised,HIGH (label lineage overlaps ClinVar/HGMD)


## 2. The circularity mechanism, concretely

Suppose REVEL was trained on a set of variants labelled *Pathogenic* that later
became ClinVar *Pathogenic* entries. Then:

- When you test "does REVEL agree with ClinVar?", the variants REVEL **memorised**
  inflate the agreement — the benchmark looks better than the tool really is.
- When REVEL **disagrees** with ClinVar on a VUS, you cannot tell whether that is
  *independent computational evidence for reclassification* (what you want) or
  simply *a variant REVEL never learned well* (label noise).

This is why the missense "413 discordances" are safest when they come from an
**unsupervised** tool. And in fact they do — the 413 list is
**AlphaMissense-vs-ClinVar** (archived integration notebook), and AlphaMissense is unsupervised.
The genuinely dangerous move would be to build the same worklist from **REVEL**
vs ClinVar and treat it as independent.

## 3. Training-data cut-off (temporal leakage)

Even an unsupervised tool is calibrated at a point in time, and supervised tools
freeze their training labels on some date. **Proper evaluation uses only ClinVar
records that were submitted *after* the tool's training freeze** — otherwise you
are testing on data the tool may have seen.

The missense-triage pipeline did **not** record or use these dates. To do it right you would:

1. Find each tool's training-data freeze (from its paper / release notes).
2. Pull ClinVar's `LastEvaluated` date per variant (it is in the raw
   `variant_summary` — the toolkit keeps `review_status`; the date column is
   available in the same file).
3. Keep only variants **first classified after** the tool's freeze → a *temporally
   held-out* test set.

Approximate freezes (⚠ **verify from each paper before using — these are rough**):

| Tool | ~Training freeze | Trained on clinical labels? |
|---|---|---|
| AlphaMissense | 2023 | No (sequence/structure) |
| EVE | 2021 | No (MSA) |
| ESM1b | 2022 | No (UniRef sequences) |
| PrimateAI | 2018 | Proxy only (primate/common) |
| REVEL | 2016 | **Yes** (HGMD + ESP) |
| SpliceAI | 2019 | No (GTEx junctions) |
| CADD | per release | Proxy (simulated vs observed) |

The key column here is the last one, not the date: a tool that never saw clinical
labels cannot leak them regardless of date.

### §3 in practice — the temporal-leakage flag table

A variant's clinical label can leak into a tool only if **the tool's training year is
after the variant was first reported pathogenic** AND the tool **learned from clinical
labels**. The classic anchor: **F508del was reported disease-causing in 1989** (Kerem et
al.), so *every* tool trained afterwards *could* have encountered it — but it only
**matters** for label-supervised tools (REVEL). Unsupervised / proxy tools (AlphaMissense,
EVE, ESM1b, SpliceAI, Pangolin, CADD) carry only **indirect** risk (benchmarks, frequency
calibration), so a date flag there is a weak caveat, not a disqualifier.

The table below is built from `tk.TOOL_YEAR` + `tk.LABEL_SUPERVISED` (single source of
truth in `toolkit.py`) and a small curated set of first-reported-pathogenic years.

In [3]:
# first-reported-pathogenic year for famous CF variants (curated from the literature;
# extend as needed). Anchor: F508del 1989.
VARIANT_REPORTED = {
    "F508del": 1989, "G551D": 1990, "G542X": 1990, "R553X": 1990, "621+1G>T": 1990,
    "R117H": 1991, "N1303K": 1991, "W1282X": 1992, "A455E": 1993, "3849+10kb C>T": 1994,
}

rows = []
for tool, ty in tk.TOOL_YEAR.items():
    sup = tk.LABEL_SUPERVISED[tool]
    for var, vy in VARIANT_REPORTED.items():
        after = ty > vy
        if after and sup:
            flag = "\u26a0 DIRECT leakage risk"      # label-supervised, trained after report
        elif after:
            flag = "indirect only"                    # unsupervised/proxy trained after
        else:
            flag = "clear (tool predates report)"
        rows.append({"tool": tool, "tool_year": ty, "label_supervised": sup,
                     "variant": var, "reported": vy, "flag": flag})

leak = pd.DataFrame(rows)
# headline: which tool x variant pairs are a DIRECT leakage risk?
direct = leak[leak["flag"].str.startswith("\u26a0")]
print(f"DIRECT leakage-risk tool x variant pairs: {len(direct)} "
      f"(all from label-supervised tools: {sorted(direct['tool'].unique())})")
# show the flag matrix for one representative variant + the REVEL row
leak.pivot_table(index="variant", columns="tool", values="flag", aggfunc="first")[
    ["REVEL", "AlphaMissense", "CADD"]]

DIRECT leakage-risk tool x variant pairs: 10 (all from label-supervised tools: ['REVEL'])


tool,REVEL,AlphaMissense,CADD
variant,,,
3849+10kb C>T,⚠ DIRECT leakage risk,indirect only,indirect only
621+1G>T,⚠ DIRECT leakage risk,indirect only,indirect only
A455E,⚠ DIRECT leakage risk,indirect only,indirect only
F508del,⚠ DIRECT leakage risk,indirect only,indirect only
G542X,⚠ DIRECT leakage risk,indirect only,indirect only
G551D,⚠ DIRECT leakage risk,indirect only,indirect only
N1303K,⚠ DIRECT leakage risk,indirect only,indirect only
R117H,⚠ DIRECT leakage risk,indirect only,indirect only
R553X,⚠ DIRECT leakage risk,indirect only,indirect only


## 4. Is CFTR2 a *safer* benchmark than ClinVar? Partially — not independent

A common move is to judge sequence predictors against a **different kind** of evidence:
CFTR2, whose calls fold in *in-vitro* CFTR **functional-assay** data (chloride
conductance, processing) plus patient outcomes. That functional axis **is** evidence
ClinVar largely lacks, so CFTR2 is **partially more orthogonal**.

**But CFTR2 is NOT an independent gold standard**, and treating "predictor vs CFTR2" as
circularity-free is a mistake:

- CFTR2's **patient/clinical** component overlaps the same evidence that feeds ClinVar.
- **ClinVar entries cite CFTR2**, and **CFTR2 informs the ACMG CFTR variant-testing
  guidance** that ClinVar submitters follow — the two databases **cross-reference**.
- So a variant's CFTR2 call and its ClinVar call are **correlated by construction**, not
  two independent opinions.

**Bottom line:** lean on CFTR2's *functional* measurements as partial orthogonal
evidence, but do not report "agrees with CFTR2" as if it were independent of ClinVar.

In [4]:
demo = tk._demo_frame().copy()
# truth = CFTR2 functional class collapsed to path/benign
def cftr2_truth(c):
    c = str(c)
    if "CF-causing" in c: return "pathogenic"
    if "Not CF" in c:     return "benign"
    return None                      # VUS / unknown -> excluded from a benchmark
demo["truth"] = demo["cftr2_class"].apply(cftr2_truth)

for tool, col in [("am","am_score"),("eve","eve_score"),("esm1b","esm1b_score")]:
    demo[tool+"_call"] = demo[col].apply(lambda s: tk.call_from_score(s, tool))

bench = demo.dropna(subset=["truth"])[
    ["protein_variant","cftr2_class","truth","am_call","eve_call","esm1b_call"]]
bench.reset_index(drop=True)

,protein_variant,cftr2_class,truth,am_call,eve_call,esm1b_call
0,G551D,CF-causing,pathogenic,pathogenic,pathogenic,pathogenic
1,F508del,CF-causing,pathogenic,na,na,na
2,R117H,CF-causing (mild),pathogenic,uncertain,benign,benign
3,R334W,CF-causing,pathogenic,pathogenic,pathogenic,pathogenic
4,G85E,CF-causing,pathogenic,pathogenic,pathogenic,pathogenic
5,R668C,Not CF-causing,pathogenic,benign,benign,benign
6,M470V,Not CF-causing,pathogenic,benign,benign,benign


In [5]:
# concordance of each unsupervised tool vs the CFTR2 functional truth (demo)
def concordance(call_col):
    ok = ((bench["truth"]=="pathogenic") & (bench[call_col]=="pathogenic")) | \
         ((bench["truth"]=="benign")     & (bench[call_col]=="benign"))
    scored = bench[call_col].isin(["pathogenic","benign"])
    return f"{ok.sum()}/{scored.sum()} concordant (of {len(bench)} truth variants)"
for t in ["am_call","eve_call","esm1b_call"]:
    print(f"{t:12s}: {concordance(t)}")
print("\n(DEMO data — this is the METHOD, not a real benchmark result.)")

am_call     : 3/5 concordant (of 7 truth variants)
eve_call    : 3/6 concordant (of 7 truth variants)
esm1b_call  : 3/6 concordant (of 7 truth variants)

(DEMO data — this is the METHOD, not a real benchmark result.)


## 5. A de-circularization checklist

Use this before claiming a predictor "disagrees with the clinical call":

1. **Is the predictor supervised on clinical labels?** If yes (REVEL), do not treat its
   ClinVar disagreement as independent evidence — or down-weight it.
2. **Temporal leakage:** was the variant reported pathogenic *before* the tool's training
   year? If so, a label-supervised tool may have memorised it (see the table in §3).
   Prefer ClinVar records submitted *after* the tool's training freeze.
3. **Prefer unsupervised tools** (AlphaMissense / EVE / ESM1b) when benchmarking against
   ClinVar — they never saw clinical labels.
4. **"Orthogonal" truth is only partial:** CFTR2's *functional assays* add real
   independent signal, but CFTR2 and ClinVar cross-cite — it is **not** an independent
   gold standard (§4).
5. **Consensus ≠ independence:** "≥3/5 tools agree" is not five independent votes if the
   tools share training data or a benchmark lineage.

## Key takeaways

- "Predictor disagrees with ClinVar" is only *evidence* if the predictor never learned
  from ClinVar-lineage labels. **REVEL is the one to distrust here; AlphaMissense / EVE /
  ESM1b are the safe ones;** CADD is proxy-trained (observed-vs-simulated), not on
  clinical labels.
- **Temporal leakage:** a variant reported pathogenic before a label-supervised tool's
  training year (e.g. **F508del, 1989**) can be memorised — flag those tool×variant pairs
  (§3) rather than counting them as validation.
- **CFTR2 is only partially orthogonal** — its functional assays add independent signal,
  but it cross-references ClinVar and is not an independent gold standard.
- This notebook is a **reference**, not a worklist: attach these caveats to the scores in
  notebooks 02–11 and to the future `predict/` runs.